### dim_customer_segment

In [0]:
# Gold Notebook: dim_customer_segment (Streaming Version)

from pyspark.sql.functions import col, count, avg, round, concat_ws

# Paths

checkpoint_path = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/dim_customer_segment/"

# ── Step 1: Read from Silver (Streaming) ───────────────────────────────

df_cust = (
spark.readStream
.format("delta")
.table("mini_project_team_a3t7.silver.customers")
)

# ── Step 2: Build Customer Segments ────────────────────────────────────

dim_seg = (
df_cust
.groupBy("age_group", "loyalty_tier", "gender")
.agg(
count("*").alias("customer_count"),
round(avg("total_spend"), 2).alias("avg_total_spend")
)
.withColumn(
"segment_key",
concat_ws("_", col("age_group"), col("loyalty_tier"), col("gender"))
)
)

# ── Step 3: Write to Gold (Streaming) ──────────────────────────────────

query = (
dim_seg.writeStream
.format("delta")
.outputMode("complete")   # required for aggregations
.option("checkpointLocation", checkpoint_path)
.trigger(availableNow=True)
.toTable("mini_project_team_a3t7.gold.dim_customer_segment")
)




In [0]:
from pyspark.sql import functions as F

# Read the Silver Customer Master table
df_cust_silver = spark.table("mini_project_team_a3t7.silver.customers")

dim_customer = (
    df_cust_silver
    .select(
        "customer_id",
        "age_group",
        "gender",
        "zip_code",
        "loyalty_tier",
        "first_purchase_date",
        "preferred_categories"
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

# Write to Gold
dim_customer.write.format("delta").mode("append").saveAsTable("mini_project_team_a3t7.gold.dim_customer_segment_new")

In [0]:
# Read the Silver POS Transactions table
import pyspark.sql.functions as F
df_pos_silver = spark.table("mini_project_team_a3t7.silver.pos_transactions")


dim_pos = (
    df_pos_silver
    .select(
        "transaction_id",
        "payment_method",
        "channel"
    )
    .dropDuplicates(["transaction_id"])
    .withColumn("_gold_processed_at", F.current_timestamp())
)

# Write to Gold
dim_pos.write.format("delta").mode("overwrite").saveAsTable("mini_project_team_a3t7.gold.dim_pos_transactions")

In [0]:
# Load the dimensions we just created
dim_cust = spark.table("mini_project_team_a3t7.gold.dim_customer_segment_new")
dim_pos_meta = spark.table("mini_project_team_a3t7.gold.dim_pos_transactions")
raw_txns = spark.table("mini_project_team_a3t7.silver.pos_transactions")

fact_sales = (
    raw_txns.alias("tx")
    # Join with Customer Dim to get Age/Loyalty
    .join(dim_cust.alias("c"), "customer_id", "inner")
    # Join with POS Dim to get Payment/Channel
    .join(dim_pos_meta.alias("p"), "transaction_id", "inner")
    
    .select(
        "tx.transaction_id",
        "tx.customer_id",
        "tx.store_id",
        "tx.product_id",
        "tx.event_timestamp",
        "tx.quantity",
        "tx.unit_price",
        "tx.total_amount",
        "c.age_group",
        "c.loyalty_tier",
        "p.channel",
        "p.payment_method"
    )
    .withColumn("inventory_date", F.to_date("event_timestamp"))
    .withColumn("_fact_processed_at", F.current_timestamp())
)

# Write to Gold (Partitioned by date for faster BI queries)
(fact_sales.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("inventory_date")
    .saveAsTable("mini_project_team_a3t7.gold.fact_customer_sales")
)
